# 5) Normalize + bound scores (fit on DEV, apply to TEST), write evaluation-ready caches
# Produces:
#   data/dtw_cache/<split>/dtw_cache_<split>_<case>.parquet
#   data/dtw_cache/normalization_params.json

In [ ]:
# 5.1) Imports & setup
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

# Ensure project root is on PYTHONPATH
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

CACHE_DIR = project_root / "data" / "dtw_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ("dev", "test")
CASES  = ("genuine", "skilled", "random")

# Features to derive and normalize (all "higher ⇒ more genuine")
FEATURE_SPECS = [
    ("s_raw",      lambda df: -df["d_raw"].astype(float)),
    ("s_bound",    lambda df: -df["d_bound"].astype(float)),
    ("s_by_path",  lambda df: -df["d_raw"].astype(float) / np.maximum(df["path_len"].astype(float), 1e-12)),
    ("s_by_avglen",lambda df: -df["d_raw"].astype(float) / np.maximum(((df["len_A"].astype(float)+df["len_B"].astype(float))/2.0), 1e-12)),
]

# The *default* score column to privilege in downstream (kept in the file for convenience)
DEFAULT_SCORE_COL = "s_bound_norm"

# %%
# 5.2) Helpers

def _find_input_cache(split: str, case: str) -> Path:
    """
    Prefer the raw cache from step 03 (dtw_<split>_<case>.parquet);
    fall back to already-normalized (dtw_cache_<split>_<case>.parquet) if present.
    """
    p1 = CACHE_DIR / split / f"dtw_{split}_{case}.parquet"
    p2 = CACHE_DIR / split / f"dtw_cache_{split}_{case}.parquet"
    if p1.exists():
        return p1
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Missing cache for {split}/{case}: tried {p1.name} and {p2.name}")

def _load(split: str, case: str) -> pd.DataFrame:
    p = _find_input_cache(split, case)
    df = pd.read_parquet(p, engine="pyarrow")
    # integrity
    need = {"pair_id","d_raw","d_bound","path_len","len_A","len_B"}
    missing = need - set(df.columns)
    if missing:
        raise KeyError(f"{split}/{case}: missing columns {missing} in {p}")
    if not df["pair_id"].is_unique:
        raise AssertionError(f"{split}/{case}: duplicate pair_id detected in {p}")
    return df

def _compute_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for name, fn in FEATURE_SPECS:
        out[name] = fn(out)
        # clean inf/nan → median
        col = out[name].astype(float)
        med = float(np.nanmedian(col))
        out[name] = np.where(np.isfinite(col), col, med)
    return out

def _fit_minmax(x: np.ndarray, lo=0.05, hi=0.95) -> tuple[float,float]:
    assert 0 < lo < hi < 1
    p_lo, p_hi = np.quantile(x.astype(float), [lo, hi])
    # guard degenerate
    if not np.isfinite(p_lo): p_lo = float(np.nanmin(x))
    if not np.isfinite(p_hi): p_hi = float(np.nanmax(x))
    if p_hi <= p_lo:
        # widen slightly
        eps = 1e-9
        p_hi = p_lo + eps
    return float(p_lo), float(p_hi)

def _apply_minmax(x: np.ndarray, p_lo: float, p_hi: float) -> np.ndarray:
    z = (x - p_lo) / max(p_hi - p_lo, 1e-12)
    return np.clip(z, 0.0, 1.0)

# %%
# 5.3) Load all caches
data = { split: { case: _load(split, case) for case in CASES } for split in SPLITS }

# %%
# 5.4) Derive features
for split in SPLITS:
    for case in CASES:
        data[split][case] = _compute_features(data[split][case])

# %%
# 5.5) Fit normalization on DEV only (per feature, per case), apply to DEV & TEST
norm_params = {}  # {feature: {case: {"p05":..., "p95":...}}}

for name, _ in FEATURE_SPECS:
    norm_params[name] = {}
    for case in CASES:
        x_dev = data["dev"][case][name].to_numpy(float)
        p05, p95 = _fit_minmax(x_dev, lo=0.05, hi=0.95)
        norm_params[name][case] = {"p05": p05, "p95": p95}

# Apply
for split in SPLITS:
    for case in CASES:
        for name, _ in FEATURE_SPECS:
            p = norm_params[name][case]
            data[split][case][f"{name}_norm"] = _apply_minmax(data[split][case][name].to_numpy(float),
                                                             p["p05"], p["p95"])

# Default score column (handy for browsing/plots; evaluation module
# will still compute its own negative distances if you choose)
for split in SPLITS:
    for case in CASES:
        if DEFAULT_SCORE_COL in data[split][case].columns:
            data[split][case]["score_default"] = data[split][case][DEFAULT_SCORE_COL].astype(float)

# %%
# 5.6) Write enriched, evaluation-ready caches
outputs = {}
for split in SPLITS:
    outputs[split] = {}
    (CACHE_DIR / split).mkdir(parents=True, exist_ok=True)
    for case in CASES:
        out_path = CACHE_DIR / split / f"dtw_cache_{split}_{case}.parquet"
        outputs[split][case] = out_path
        # keep original cache columns + new features
        cols_order = [
            # identifiers
            "pair_id",
            # labels (if present)
            *([ "label" ] if "label" in data[split][case].columns else []),
            # original cache
            "d_raw","d_bound","path_len","len_A","len_B","backend","window","mode",
            # provenance (if present)
            *([ "split","case" ] if {"split","case"}.issubset(data[split][case].columns) else []),
            # derived features
            "s_raw","s_bound","s_by_path","s_by_avglen",
            "s_raw_norm","s_bound_norm","s_by_path_norm","s_by_avglen_norm",
            # convenience
            *([ "score_default" ] if "score_default" in data[split][case].columns else []),
        ]
        # filter to those that actually exist
        cols_order = [c for c in cols_order if c in data[split][case].columns]
        data[split][case].to_parquet(out_path, index=False, engine="pyarrow", compression="zstd")
        # quick integrity
        dfc = pd.read_parquet(out_path, engine="pyarrow")
        assert dfc["pair_id"].is_unique, f"{split}/{case}: duplicate pair_id in written cache"
        print(f"✅ wrote {out_path}")

# %%
# 5.7) Persist normalization params
params_path = CACHE_DIR / "normalization_params.json"
params_path.write_text(json.dumps(norm_params, indent=2))
print(f"📝 wrote {params_path}")

# %%
# 5.8) Spot-check summary
for split in SPLITS:
    print(f"\n=== SUMMARY: {split.upper()} ===")
    for case in CASES:
        df = data[split][case]
        print(f"{case:8s} rows={len(df):6d} | "
              f"{DEFAULT_SCORE_COL} mean={df.get(DEFAULT_SCORE_COL, pd.Series([np.nan])).mean():.3f}")